# Pipeline RAG — Retrieval Augmented Generation

## Comment ça fonctionne ?

Un pipeline RAG se déroule en **3 étapes** à chaque question :

```
Question utilisateur
      ↓
[1. RETRIEVAL] — on transforme la question en vecteur (embedding),
                 on cherche les N événements les plus proches dans FAISS
      ↓
[2. AUGMENTATION] — on injecte ces événements dans un prompt contextualisé
      ↓
[3. GENERATION] — on envoie le prompt à Mistral qui génère la réponse finale
```

**Pourquoi ne pas juste envoyer la question à Mistral directement ?**  
Mistral ne connaît pas les événements de Rennes — ses données d'entraînement sont figées dans le temps.  
Le RAG lui donne accès à nos données en temps réel, sans le ré-entraîner.

## Ce qu'on va construire

1. Charger l'index FAISS et le mapping
2. Tester la recherche de documents similaires (retrieval)
3. Construire le prompt augmenté
4. Générer une réponse avec Mistral
5. Assembler le pipeline complet

In [9]:
import os
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage

embed_model = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

faiss_store = FAISS.load_local("../data/faiss_index", embed_model, allow_dangerous_deserialization=True)
client = MistralClient(api_key=os.getenv("MISTRAL_API_KEY"))

print(f"Index chargé : {faiss_store.index.ntotal} vecteurs")

Index chargé : 5028 vecteurs


## 1. Retrieval — recherche des documents pertinents

On transforme la question en vecteur avec le **même modèle d'embedding** utilisé lors de l'indexation (`mistral-embed`).  
C'est crucial : si on embeddait avec un modèle différent, les vecteurs ne seraient pas comparables.

FAISS calcule ensuite la distance euclidienne entre ce vecteur et tous les vecteurs de l'index,  
et retourne les `k` plus proches — ce sont les événements les plus "sémantiquement proches" de la question.

In [10]:
question = "Quels concerts sont prévus à Rennes ?"

# similarity_search embede la question et retourne les k Documents les plus proches
docs = faiss_store.similarity_search(question, k=5)

print(f"Top 5 résultats pour : '{question}'\n")
for i, doc in enumerate(docs):
    print(f"{i+1}. {doc.metadata['title']}")

Top 5 résultats pour : 'Quels concerts sont prévus à Rennes ?'

1. Le concert du mardi - 𝗘́𝗹𝗶𝘀𝗮 𝗰𝗵𝗮𝗻𝘁𝗲 𝗝𝗲𝗮𝗻𝗻𝗲
2. Concert Jelias
3. Le Grand Concert de Noël
4. Le concert du mardi - 𝗔𝗿𝗼̂𝗺𝗲(𝘀)
5. KEREN ANN EN CONCERT


## 2. Augmentation — construction du prompt

On injecte les documents récupérés dans un prompt structuré.  
Le prompt dit à Mistral : "voici des informations sur des événements, réponds à la question en te basant uniquement sur ces informations".

Cette contrainte ("uniquement sur ces informations") est importante pour la **faithfulness** — une des métriques RAGAS qu'on évaluera plus tard. Elle évite que Mistral invente des événements qui n'existent pas.

In [11]:
context_parts = []
for doc in docs:
    m = doc.metadata
    context_parts.append(
        f"Titre : {m['title']}\n"
        f"Description : {doc.page_content}\n"
        f"Tarif : {m['conditions']}\n"
        f"Dates : {m['date_start']} → {m['date_end']}\n"
        f"Lieu : {m['location']}\n"
        f"Lien : {m['url']}"
    )
context = "\n\n---\n\n".join(context_parts)

prompt = f"""Tu es un assistant spécialisé dans les événements culturels à Rennes.
Réponds à la question en te basant uniquement sur les événements fournis ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.

ÉVÉNEMENTS :
{context}

QUESTION : {question}

RÉPONSE :"""

print(prompt)

Tu es un assistant spécialisé dans les événements culturels à Rennes.
Réponds à la question en te basant uniquement sur les événements fournis ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.

ÉVÉNEMENTS :
Titre : Le concert du mardi - 𝗘́𝗹𝗶𝘀𝗮 𝗰𝗵𝗮𝗻𝘁𝗲 𝗝𝗲𝗮𝗻𝗻𝗲
Description : Titre : Le concert du mardi - 𝗘́𝗹𝗶𝘀𝗮 𝗰𝗵𝗮𝗻𝘁𝗲 𝗝𝗲𝗮𝗻𝗻𝗲
Description : Élisa Chante Jeanne “Élisa Chante Jeanne” est un quartet swing en hommage à Jeanne Moreau. Des chansons écrites par Serge Rezvani, alias Cyrus Bassiak, le poète Norge et Jeanne Moreau elle-même. C'est avec beaucoup d'émotion que vous baladeront les chansons écrites par ou pour Jeanne et chantées par Élisa. Vous connaissez sans doute les célèbres Le Tourbillon De La Vie et J’ai La Mémoire Qui Flanche , qui sauront vous faire chanter en chœur. Vous découvrirez aussi des pépites cachées de la chanson française. Distribution : Elisa Robin - Chant Olivier Roth - Guitare Bérenger Heurtaux - Guitare Lyn Aubert - Contrebasse 1er set : 18

## 3. Génération — réponse de Mistral

On envoie le prompt augmenté à Mistral qui génère la réponse finale.  
On utilise `mistral-small-latest` — un bon compromis qualité/coût pour la génération de texte.

In [12]:
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage

client = MistralClient(api_key=os.getenv("MISTRAL_API_KEY"))

response = client.chat(
    model="mistral-small-latest",
    messages=[ChatMessage(role="user", content=prompt)]
)

print(response.choices[0].message.content)

Voici les concerts prévus à Rennes selon les événements fournis :

1. **Le concert du mardi - Élisa chante Jeanne**
   - **Date** : 22 juillet 2025 (16h-18h)
   - **Lieu** : À proximité de la place Eugène Aulnette (1er set à 18h devant la Maison du Parc, 2e set à 19h place Eugène Aulnette)
   - **Tarif** : Entrée libre
   - **Description** : Concert swing en hommage à Jeanne Moreau, avec des chansons de Serge Rezvani, Norge et Jeanne Moreau.

2. **Concert Jelias**
   - **Date** : 7 novembre 2025 (17h30-18h15)
   - **Lieu** : Hors les murs, Les Champs Libres (10 Cours des Alliés)
   - **Tarif** : Gratuit
   - **Description** : Groupe pop/rock/jazz/funk de la ville jumelle Erlangen (Allemagne), interprétant les chansons de Julius.

3. **Le concert du mardi - Arôme(s)**
   - **Date** : 8 juillet 2025 (16h-17h30)
   - **Lieu** : Pump track du parc de Quincé (rue Aurélie Nemours)
   - **Tarif** : Entrée libre
   - **Description** : DJ-set et percussions mêlant musiques électroniques africai

## 4. Pipeline complet — fonction `ask()`

On regroupe les 3 étapes en une seule fonction réutilisable.  
C'est cette fonction qui sera importée par l'API FastAPI.

In [ ]:
def ask(question: str, k: int = 5) -> tuple[str, list[str]]:
    """Pipeline RAG complet : retrieval → augmentation → génération.
    Retourne (réponse générée, liste des corpus récupérés).
    """
    # 1. Retrieval
    docs = faiss_store.similarity_search(question, k=k)
    contexts = [doc.page_content for doc in docs]
    context_parts = []
    for doc in docs:
        m = doc.metadata
        context_parts.append(
            f"Titre : {m['title']}\n"
            f"Description : {doc.page_content}\n"
            f"Tarif : {m['conditions']}\n"
            f"Dates : {m['date_start']} → {m['date_end']}\n"
            f"Lieu : {m['location']}\n"
            f"Lien : {m['url']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # 2. Augmentation
    prompt = f"""Tu es un assistant spécialisé dans les événements culturels à Rennes.
Réponds à la question en te basant uniquement sur les événements fournis ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.

ÉVÉNEMENTS :
{context}

QUESTION : {question}

RÉPONSE :"""

    # 3. Génération
    response = client.chat(
        model="mistral-small-latest",
        messages=[ChatMessage(role="user", content=prompt)]
    )
    return response.choices[0].message.content, contexts


answer, _ = ask("Y a-t-il des expositions gratuites à Rennes ?")
print(answer)

## 5. Test hors contexte

On vérifie que le modèle respecte la contrainte du prompt et ne répond pas en dehors des événements fournis.

In [ ]:
answer, _ = ask("Quelle est la capitale de l'Allemagne ?")
print(answer)